[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/submission-yy726/notebooks/12_dpo_unsloth.ipynb)

# 12 · DPO on Qwen3 Base（Unsloth + W&B）——从共享 SFT 检查点 S1 开始

**先跑 notebook 15。** 本 notebook 默认 `INIT_MODEL="auto"`（15 推到 HF 的 `<你的用户名>/qwen3-4b-s1-sft`），做完 DPO 后把合并模型推到
`<你的用户名>/qwen3-4b-d1-dpo`（私有）；notebook 11 的 `INIT_MODEL` 填它就能接着做 GRPO（OLMo 3：RL 从 DPO 检查点开始比从 SFT 开始更好）。

结构参考 Unsloth 官方 [Zephyr (7B) DPO](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Zephyr_(7B)-DPO.ipynb)：
`PatchDPOTrainer()` + TRL `DPOTrainer(ref_model=None)`（LoRA 时参考模型 = 关掉 adapter 的模型）。

**偏好数据**（`DPO_DATA="both"`，依据见 `experiments/DATASETS.md`）：
- `dolci`：OLMo 3 公开的 [`allenai/Dolci-Instruct-DPO`](https://huggingface.co/datasets/allenai/Dolci-Instruct-DPO)（26 万对；赛前公开，可以上榜）。
  抽样统计：**OpenThoughts3 科学题占 11%（约 2.9 万对）**，chosen 中位数约 1100 token、rejected 约 740；其中约 1/4 是 delta learning 对
  （chosen = Qwen3-32B，rejected = Qwen3-0.6B），其余是 LLM 判分挑出来的"强模型 vs 弱模型"对。两种都用（`DOLCI_DPO_TYPES=None`，X1b 之前只用了 delta 对）。
  默认取科学题 + Tulu 3 数学题。OLMo 3 Instruct 做完 DPO：GPQA 30.2 → 34.6–35.5。
- `onpolicy`：留出的 4 选 1 科学题（和 GPQA 同一种题面，S1 没见过解答），每题让**当前模型**采样 4 个回答；chosen = 答对且正常收尾，rejected = 答错。
  这是直接针对"GPQA 题面下答对"的信号。
- 可选 `allenai/Dolci-Think-DPO-7B`（Qwen3-32B 思考 vs 0.6B 思考；OLMo 3 Think：45.1 → 48.4），但科学题 chosen 中位数约 4700 token、数学约 7700，太长。

**损失**：`LOSS_TYPE="sigmoid_norm"`——OLMo 3 / Tülu 3 用的长度归一化 DPO（按回答的 token 数平均 log 概率比，β=5），
避免模型靠"写长"取胜（Dolci 的 chosen 有 74% 比 rejected 长）。学习率 1e-5（OLMo 3 全参 1e-6，LoRA 放大 10 倍）。

| 模式 | 起点 | 在线采样 | DPO | 评测 | 时间 |
|---|---|---|---|---|---|
| `quick` | 0.6B（15 的 quick 模式，或 `INIT_MODEL=None` 直接用 base） | 200 题 × 4 | 约 200 + 少量对，≤1536 token | Diamond 前 60 题 | T4 约 20 分钟 |
| `a100` | 4B 的 S1 | 1500 题 × 4（约 20 分钟） | 约 6000 对 Dolci + 约 1500 对在线，≤4096 token，1 epoch | Diamond + Main，≤8192，4 次采样 | 约 1.5–2 小时（估） |

In [1]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

name, memory.total [MiB], compute_cap
NVIDIA L4, 23034 MiB, 8.9


In [2]:
# 克隆提交分支（数据准备 / 去污染 / 共用代码 scripts/recipe_utils.py 都在里面）
import os, sys
if not os.path.exists("GPQA-eval"):
    !git clone -q -b submission-yy726 https://github.com/yy726/GPQA-eval.git
if os.path.basename(os.getcwd()) != "GPQA-eval":   # 重跑这一格时不要再进一层
    %cd GPQA-eval
!git pull -q
if os.path.abspath("scripts") not in sys.path:
    sys.path.insert(0, os.path.abspath("scripts"))
import recipe_utils
print("reloaded:", recipe_utils.reload_scripts())   # 改了 scripts/ 之后重跑这一格即可，不用重启 runtime

/content/GPQA-eval
reloaded: ['recipe_utils']


In [3]:
# 安装（约 3–5 分钟）。输出不隐藏：如果安装报错请看这里。
!pip install --upgrade -q uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install unsloth vllm wandb {_numpy} {_pil}

Using Python 3.13.16 environment at: /usr
Resolved 228 packages in 1.62s
Prepared 92 packages in 18.69s
Uninstalled 13 packages in 979ms
Installed 92 packages in 270ms
 + agent-detector==2.0.0
 + anthropic==2.9.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + bitsandbytes==0.50.2
 + blake3==1.0.11
 + cbor2==6.3.0
 + compressed-tensors==0.17.0
 - cuda-core==0.7.0
 + cuda-core==1.3.0
 + cuda-tile==1.8.1
 + cut-cross-entropy==25.1.1
 + depyf==0.20.0
 + detect-installer==0.2.1
 + dnspython==2.9.2
 + email-validator==2.4.0
 - fastapi==0.141.1
 + fastapi==0.136.3
 + fastapi-cli==0.0.34
 + fastapi-cloud-cli==0.28.0
 + fastar==0.12.0
 + fastsafetensors==0.5.0
 + flashinfer-python==0.7.0.post1
 + hf-transfer==0.1.9
 + humming-kernels==0.1.16
 + ijson==3.6.0
 + instanttensor==0.2.3
 + interegular==0.3.3
 + jmespath==1.1.0
 + json5==0.17.3
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.7.6
 - llvmlite==0.46.0
 + llvmlite==0.47.0
 + lm-format-enforcer==0.11.3
 + loguru==0.8.0
 + mcp==2.3.0
 + mcp-ty

In [4]:
import os, importlib.metadata as md, torch
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
# T4（计算能力 7.5）不支持 FlashInfer 采样器（A100 不受影响）
if torch.cuda.get_device_capability()[0] < 8:
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
    print("T4 detected: FlashInfer disabled for vLLM")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
VERSIONS = {}
for p in ["unsloth", "unsloth_zoo", "vllm", "trl", "transformers", "torch", "peft", "wandb"]:
    try: VERSIONS[p] = md.version(p)
    except md.PackageNotFoundError: VERSIONS[p] = None
print(GPU_NAME, GPU_MEM_GB, "GB"); print(VERSIONS)

NVIDIA L4 22.0 GB
{'unsloth': '2026.11.5', 'unsloth_zoo': '2026.11.5', 'vllm': '0.31.0', 'trl': '1.15.0', 'transformers': '5.17.0', 'torch': '2.13.0', 'peft': '0.21.2', 'wandb': '0.28.1'}


In [5]:
# Colab 左侧 🔑 Secrets：HF_TOKEN（必需；先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款）
#                      WANDB_API_KEY（可选；没有就不记录 W&B）
for _k in ["HF_TOKEN", "WANDB_API_KEY"]:
    try:
        from google.colab import userdata
        _v = userdata.get(_k)
        if _v:
            os.environ[_k] = _v
    except Exception:
        pass
from huggingface_hub import login, whoami
login(token = os.environ.get("HF_TOKEN"))
HF_USER = whoami(token = os.environ.get("HF_TOKEN"))["name"]
print("HF user:", HF_USER, "|", "W&B key found" if os.environ.get("WANDB_API_KEY") else "No WANDB_API_KEY: W&B logging disabled")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF user: darcyyao | W&B key found


## 0. 配置

In [6]:
EXP_NAME     = "D1"
MODE         = "a100"                         # "quick" | "a100"
MODEL_NAME   = "unsloth/Qwen3-4B-Base"        # quick: "unsloth/Qwen3-0.6B-Base"
BASE_MODEL   = "Qwen/Qwen3-4B-Base"
INIT_MODEL   = "auto"   # "auto" = notebook 15 的共享 SFT 检查点 f"{HF_USER}/qwen3-4b-s1-sft"；也可以填别的仓库或本地路径；None = base
PUSH_REPO    = "auto"   # "auto" = f"{HF_USER}/qwen3-4b-d1-dpo"（私有）：DPO 后合并推到 HF，notebook 11 的 INIT_MODEL 填它；None = 只存本地
LOAD_IN_4BIT = False
CHAT_STYLE   = "olmo"   # 和 notebook 11 一致：回答以 <|endoftext|> 结束
GEN_BUDGET   = None     # 评测的生成预算。None = 预设（a100: 8192，和排行榜一致）；16384 = 只在参考分支放宽（从思维链检查点开始时用）
USE_DRIVE    = True     # adapter、每一轮的结果、评测结果都存到 Google Drive：runtime 重置后不会丢
RESUME       = True     # 已经完成的评测和 DPO 轮次直接读结果，不重跑；False = 全部重来
AUTO_DISCONNECT = True  # 全部跑完后自动断开 runtime（不再消耗 CU）。想留着调试就设 False

DPO_DATA         = "both"            # "both" | "dolci"（Dolci-Instruct-DPO 的强 vs 弱偏好对）| "onpolicy"（自己采样的答对 vs 答错）
PROMPT_SOURCE    = "science"         # 在线偏好对的题库："science"（MegaScience 留出的 4 选 1 科学题）| "mmlu_pro_stem" | "sciq"
ROUNDS           = 1                 # 2 = 迭代 DPO（第二轮用第一轮之后的模型重新采样）
PAIRS_PER_PROMPT = 2                 # 每题最多几对

LOSS_TYPE  = "sigmoid_norm"          # 长度归一化 DPO（OLMo 3 / Tülu 3）；"sigmoid" = 标准 DPO
DPO_BETA   = None                    # None = sigmoid_norm 用 5.0（OLMo 3），其它用 0.1
DPO_LR     = 1e-5                    # OLMo 3 Instruct 全参 1e-6；LoRA 放大 10 倍
SFT_WEIGHT = None                    # chosen 上加 NLL 项的权重（IRPO 的做法，只适合 DPO_DATA="onpolicy"，可试 1.0）。
                                     # Dolci 的 chosen 不要加：OLMo 3 在 chosen 上继续 SFT，GPQA 45.1 → 40.2

DOLCI_DPO_DATASET = "allenai/Dolci-Instruct-DPO"   # 或 "allenai/Dolci-Think-DPO-7B"（长，过滤后剩得少）
DOLCI_DPO_TYPES   = None                           # None = delta learning 对和 LLM 判分对都用；["delta_learning"] = 只用 32B vs 0.6B
DOLCI_DPO_INCLUDE = ["openthoughts3-full-filtered-science", "personas-math", "personas-algebra"]   # 匹配 prompt_id / dataset_source
DOLCI_DPO_EXCLUDE = ["safety", "wildguard", "wildjailbreak", "python", "code", "aya", "precise", "flan"]

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "gpqa-hackathon")
WANDB_ENTITY  = os.environ.get("WANDB_ENTITY")

PRESETS = dict(
    quick = dict(max_seq_len=2048, gpu_util=0.6, lora_rank=16, n_prompts=200, k=4, gen_max=512,
                 dolci_n=200, dpo_max_len=1536, dpo_batch=2, dpo_accum=4,
                 eval_configs=["gpqa_diamond"], eval_n=60, eval_max_tokens=1024, eval_samples=1, shuffle_buffer=1000),
    a100 = dict(max_seq_len=12288, gpu_util=0.6, lora_rank=64, n_prompts=1500, k=4, gen_max=3072,
                dolci_n=6000, dpo_max_len=4096, dpo_batch=2, dpo_accum=8,
                eval_configs=["gpqa_diamond", "gpqa_main"], eval_n=None, eval_max_tokens=8192, eval_samples=4,
                shuffle_buffer=20000),
)
Q = dict(PRESETS[MODE])
if GEN_BUDGET:
    Q.update(eval_max_tokens = GEN_BUDGET, max_seq_len = max(Q["max_seq_len"], GEN_BUDGET + 4096))
MAX_SEQ_LEN, GPU_UTIL, LORA_RANK = Q["max_seq_len"], Q["gpu_util"], Q["lora_rank"]
from recipe_utils import resolve_init_model, sft_repo_name
INIT_MODEL = resolve_init_model(INIT_MODEL, HF_USER, MODEL_NAME)
if PUSH_REPO == "auto":
    PUSH_REPO = sft_repo_name(HF_USER, MODEL_NAME).replace("-s1-sft", f"-{EXP_NAME.lower()}-dpo")
if INIT_MODEL is None:
    print("[!] INIT_MODEL 为空：直接从 base 做 DPO（base 不会收尾，在线偏好对基本采不到）。建议先跑 notebook 15。")
from recipe_utils import persist_dir, StageCache
PERSIST = persist_dir(f"{EXP_NAME}_{MODE}", USE_DRIVE)     # 开头会弹一次 Drive 授权
CACHE = StageCache(PERSIST, enabled = RESUME)
CACHE.check_config(dict(model = MODEL_NAME, init_model = INIT_MODEL, chat_style = CHAT_STYLE, dpo_data = DPO_DATA,
                        prompt_source = PROMPT_SOURCE, rounds = ROUNDS, pairs_per_prompt = PAIRS_PER_PROMPT, loss_type = LOSS_TYPE,
                        beta = DPO_BETA, lr = DPO_LR, sft_weight = SFT_WEIGHT, dolci = DOLCI_DPO_DATASET, types = DOLCI_DPO_TYPES,
                        include = DOLCI_DPO_INCLUDE,
                        **{k: Q[k] for k in ("max_seq_len", "lora_rank", "n_prompts", "k", "gen_max", "dolci_n", "dpo_max_len",
                                             "dpo_batch", "dpo_accum", "eval_n", "eval_max_tokens")}))
def ADIR(name):
    return f"{PERSIST}/{name}"
print(EXP_NAME, MODE, "start from:", INIT_MODEL or MODEL_NAME, "-> push:", PUSH_REPO, Q)
print("persist dir:", PERSIST, "| finished stages:", CACHE.done())

Mounted at /content/drive
D1 a100 start from: darcyyao/qwen3-4b-s1-sft -> push: darcyyao/qwen3-4b-d1-dpo {'max_seq_len': 12288, 'gpu_util': 0.6, 'lora_rank': 64, 'n_prompts': 1500, 'k': 4, 'gen_max': 3072, 'dolci_n': 6000, 'dpo_max_len': 4096, 'dpo_batch': 2, 'dpo_accum': 8, 'eval_configs': ['gpqa_diamond', 'gpqa_main'], 'eval_n': None, 'eval_max_tokens': 8192, 'eval_samples': 4, 'shuffle_buffer': 20000}
persist dir: /content/drive/MyDrive/gpqa_ckpt/D1_a100 | finished stages: []


## 1. 加载模型（`fast_inference=True`：采样、评测、训练共用一个 vLLM 引擎）

In [7]:
import os, torch, json, random, time
if torch.cuda.get_device_capability()[0] < 8:
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
from unsloth import FastLanguageModel
try:                                   # 官方 DPO 示例要求先打补丁；新版 Unsloth 可能已自动处理
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
except ImportError:
    print("[i] PatchDPOTrainer not available in this Unsloth version (trainers are patched automatically)")
from recipe_utils import (STOP_STRS, TABLE_COLS, QuickEval, setup_chat, chat_prompt, log_eval, dolci_dpo_pair,
                          load_mcq_rows, is_correct_and_stopped, decontam_rows, n_chat_tokens, config_kwargs,
                          dpo_loss_kwargs, run_script, wandb_init, print_table, load_guard, restore_lora, end_runtime)

# 这一格可以反复运行：模型已经加载过就直接复用（vLLM 引擎不能在同一个进程里建第二次，以前重跑会报错）。
_KEY = (INIT_MODEL or MODEL_NAME, MAX_SEQ_LEN, LORA_RANK, GPU_UTIL, LOAD_IN_4BIT)
if load_guard(globals(), _KEY):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = INIT_MODEL or MODEL_NAME,
        max_seq_length = MAX_SEQ_LEN,
        load_in_4bit = LOAD_IN_4BIT,
        fast_inference = True,
        max_lora_rank = LORA_RANK,
        gpu_memory_utilization = GPU_UTIL,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r = LORA_RANK,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = LORA_RANK * 2,
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )
    _LOADED_KEY = _KEY
else:
    print("[i] 模型已经加载，直接复用（没有重新加载）。训练格开始时会把 LoRA 权重重置。")
STOP_IDS = setup_chat(tokenizer, model, style = CHAT_STYLE)   # 回答以 eos=<|endoftext|> 结束（照 OLMo 3）
print("eos:", tokenizer.eos_token, "| stop ids:", STOP_IDS, "| pad:", tokenizer.pad_token)

RUN_NAME = f"{EXP_NAME}-dpo-{MODEL_NAME.split('/')[-1]}-{MODE}"
wandb, REPORT_TO = wandb_init(WANDB_PROJECT, RUN_NAME, Q | dict(
    exp = EXP_NAME, recipe = "dpo", mode = MODE, model = MODEL_NAME, init_model = INIT_MODEL, base_model = BASE_MODEL,
    dpo_data = DPO_DATA, prompt_source = PROMPT_SOURCE, rounds = ROUNDS, pairs_per_prompt = PAIRS_PER_PROMPT,
    chat_style = CHAT_STYLE, dolci_dpo = DOLCI_DPO_DATASET, dolci_dpo_types = DOLCI_DPO_TYPES, dolci_dpo_include = DOLCI_DPO_INCLUDE,
    beta = DPO_BETA, lr = DPO_LR, loss_type = LOSS_TYPE, sft_weight = SFT_WEIGHT,
    gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB, versions = VERSIONS), WANDB_ENTITY)
REPORT = dict(notebook = "12_dpo_unsloth", exp = EXP_NAME, mode = MODE, model = MODEL_NAME, init_model = INIT_MODEL,
              dpo_data = DPO_DATA, prompt_source = PROMPT_SOURCE, rounds = ROUNDS, beta = DPO_BETA, lr = DPO_LR,
              loss_type = LOSS_TYPE, sft_weight = SFT_WEIGHT, dolci_types = DOLCI_DPO_TYPES, dolci_include = DOLCI_DPO_INCLUDE, q = Q)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1117 01:32:33.925000 9982 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1117 01:32:34.036000 9982 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.13.0+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.5: Fast Qwen3 patching. Transformers: 5.17.0. vLLM: 0.31.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.13.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Not planning a device map; vLLM places its own weights. Using `sequential`.


model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Unsloth: Standby mode is enabled. Changing `gpu_memory_utilization` to 0.78375.
Unsloth: Patched vLLM's piecewise graph-partition pass so compilation_config=3 (CUDA graphs) works with LoRA.
Unsloth: Not an error, but `level` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `use_cudagraph` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `use_inductor` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `swap_space` is not supported in vLLM. Skipping.
Unsloth: Not an error, but `device` is not supported in vLLM. Skipping.
Unsloth: vLLM loading /root/.cache/huggingface/hub/models--darcyyao--qwen3-4b-s1-sft/snapshots/4bcf7a5cf8edbafcb6ee41afc41edd3300f263ab with actual GPU utilization = 77.44%
Unsloth: Your GPU has CUDA compute capability 8.9 with VRAM = 22.03 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 4096. Num Sequences = 48.
Unsloth: vLLM's KV Cache

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


... (13 outputs left out)

Unsloth: compiling each transformer block as one region (Qwen3DecoderLayer).
eos: <|endoftext|> | stop ids: [151643, 151645, 151644, 70564] | pad: <|vision_pad|>


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: darcy-yi-yao (darcy-yi-yao-apple) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


W&B: https://wandb.ai/darcy-yi-yao-apple/gpqa-hackathon/runs/ukg2wo1d


## GPQA 快速评测（只输出聚合数字；不记录题目或模型输出）

口径和 notebook 11 完全一样（`scripts/recipe_utils.py` 的 `QuickEval`）：`flex` = 输出中最后一个 `(A)`–`(D)`（排行榜指标），`strict` = 第一个 `The answer is (X)`。
每题采样 `eval_samples` 次，`acc_flex` 是平均（avg@k），`acc_flex_se` 是标准误。**比较训练前后看汇总里的配对差值 ± 标准误**（|z| < 2 = 看不出差别）：
X1b 里单样本的 Diamond（198 题）两个阶段之差的标准误约 4 分，SFT → GRPO 的 "−6.6" 其实只有 1.4 个标准误。
⚠️ 不是排行榜分数（选项顺序与 lm-eval 不同），只用来比较同一次运行里训练前后的变化。

In [8]:
from vllm import SamplingParams

EVAL = QuickEval.from_gpqa(tokenizer, Q["eval_configs"], Q["eval_n"], max_tokens = Q["eval_max_tokens"],
                           max_model_len = MAX_SEQ_LEN, samples = Q["eval_samples"], stop_ids = STOP_IDS)
RESULTS = EVAL.results

def quick_eval(tag, lora_request = None, samples = None):
    r = EVAL.run(tag, lambda texts, kw: model.fast_generate(texts, sampling_params = SamplingParams(**kw),
                                                             lora_request = lora_request), samples, cache = CACHE)
    log_eval(wandb, r, tag)

if INIT_MODEL:
    quick_eval("0_init")
else:
    quick_eval("0_base", samples = 1)   # base 不收尾，每个回答都写满预算（约 30 分钟）：只采样 1 次

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

gpqa_diamond.csv:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/198 [00:00<?, ? examples/s]

gpqa_main.csv:   0%|          | 0.00/3.21M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/448 [00:00<?, ? examples/s]

(MainProcess pid=9982) INFO 11-17 01:36:08 [scheduler.py:2691] setting pause state to UNPAUSED


... (4 outputs left out)

0_init gpqa_diamond {'n': 198, 'acc_flex': 36.5, 'acc_strict': 36.4, 'extract_rate': 92.3, 'acc_answered': 39.5, 'acc_stopped': 38.7, 'acc_truncated': 2.1, 'truncation_rate': 6.1, 'mean_tokens': 1534, 'repeat_rate': 23.7, 'answer_then_more': 0.0, 'glitch_rate': 0.0, 'letters': {'A': 206, 'B': 177, 'C': 159, 'D': 189, 'none': 61}, 'samples': 4, 'acc_flex_se': 2.4, 'by_domain': {'Biology': {'n': 19, 'acc_flex': 46.1, 'extract_rate': 98.7, 'truncation_rate': 0.0}, 'Chemistry': {'n': 93, 'acc_flex': 27.7, 'extract_rate': 96.5, 'truncation_rate': 3.5}, 'Physics': {'n': 86, 'acc_flex': 43.9, 'extract_rate': 86.3, 'truncation_rate': 10.2}}, 'minutes': 54.4}
    0_init gpqa_diamond | 给出答案的回答：答对 39.5%（没给出答案的占 7.7%）| 自己收尾的回答：答对 38.7% | 被截断的回答：答对 2.1%（占 6.1%）
    0_init gpqa_diamond | 按学科：Biology 46.1（19 题，无答案 1.3%，截断 0.0%） | Chemistry 27.7（93 题，无答案 3.5%，截断 3.5%） | Physics 43.9（86 题，无答案 13.7%，截断 10.2%）
(MainProcess pid=9982) INFO 11-17 02:30:33 [scheduler.py:2691] setting pause state to UNPAUSED


Rendering prompts:   0%|          | 0/448 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1792/1792 [1:55:44<00:00,  3.88s/it, est. speed input: 59.43 toks/s, output: 381.71 toks/s]


0_init gpqa_main {'n': 448, 'acc_flex': 35.3, 'acc_strict': 35.2, 'extract_rate': 92.2, 'acc_answered': 38.3, 'acc_stopped': 37.4, 'acc_truncated': 2.8, 'truncation_rate': 6.0, 'mean_tokens': 1479, 'repeat_rate': 22.0, 'answer_then_more': 0.0, 'glitch_rate': 0.0, 'letters': {'A': 465, 'B': 442, 'C': 375, 'D': 371, 'none': 139}, 'samples': 4, 'acc_flex_se': 1.6, 'by_domain': {'Biology': {'n': 78, 'acc_flex': 47.1, 'extract_rate': 96.2, 'truncation_rate': 2.2}, 'Chemistry': {'n': 183, 'acc_flex': 26.0, 'extract_rate': 93.3, 'truncation_rate': 6.1}, 'Physics': {'n': 187, 'acc_flex': 39.6, 'extract_rate': 89.6, 'truncation_rate': 7.5}}, 'acc_flex_diamond_part': 38.4, 'acc_flex_rest': 32.9, 'n_diamond_part': 198, 'minutes': 115.8}
    0_init gpqa_main | 给出答案的回答：答对 38.3%（没给出答案的占 7.8%）| 自己收尾的回答：答对 37.4% | 被截断的回答：答对 2.8%（占 6.0%）
    0_init gpqa_main | 按学科：Biology 47.1（78 题，无答案 3.8%，截断 2.2%） | Chemistry 26.0（183 题，无答案 6.7%，截断 6.1%） | Physics 39.6（187 题，无答案 10.4%，截断 7.5%）


## 2. 偏好数据

- 在线偏好对的题目先过 GPQA 13-gram 去污染，再按长度过滤（题目 + 回答上限 ≤ `dpo_max_len`）。
- Dolci DPO：流式读取，按来源 / domain 过滤，`chosen` / `rejected` 两段都要 ≤ `dpo_max_len`，同样去污染。

In [9]:
from datasets import load_dataset

DATA_INFO = {}
mcq = []
if DPO_DATA in ("onpolicy", "both"):
    if PROMPT_SOURCE == "science":
        # MegaScience 留出的 4 选 1 科学题（S1 没见过解答）。从题库的末尾取，notebook 11 的 GRPO 从开头取，尽量不重叠
        from science_mix import load_mcq_pool
        if not os.path.exists("data/s1/rl_mcq.jsonl"):
            run_script(["scripts/science_mix.py", "--out-dir", "data/s1", "--pools-only"], show_tail = 40)   # 下载约 1.9 GB，约 3 分钟
        mcq = [{"prompt": r["prompt"][0]["content"], "answer": r["answer"], "source": r["source"]}
               for r in load_mcq_pool("data/s1/rl_mcq.jsonl")[::-1][: Q["n_prompts"]]]
    else:
        # SciQ 前 6000 条可能已被 notebook 11 用于 SFT / GRPO，跳过
        mcq = load_mcq_rows(PROMPT_SOURCE, Q["n_prompts"], skip = 6000 if PROMPT_SOURCE == "sciq" else 0)
    mcq, DATA_INFO["onpolicy_dropped_decontam"] = decontam_rows(mcq, lambda r: {"prompt": r["prompt"]}, "dpo_prompts")
    budget = Q["dpo_max_len"] - Q["gen_max"]
    mcq = [r for r in mcq if n_chat_tokens(tokenizer, [{"role": "user", "content": r["prompt"]}], True) <= budget]
    DATA_INFO["onpolicy_prompts"] = len(mcq)

def load_dolci_dpo(n):
    ds = load_dataset(DOLCI_DPO_DATASET, split = "train", streaming = True).shuffle(seed = 0, buffer_size = Q["shuffle_buffer"])
    out, scanned = [], 0
    for ex in ds:
        if len(out) >= n or scanned >= 300_000:
            break
        if scanned == 0:
            print(DOLCI_DPO_DATASET, "columns:", sorted(ex.keys()))
        scanned += 1
        pair = dolci_dpo_pair(ex, DOLCI_DPO_INCLUDE, DOLCI_DPO_EXCLUDE, DOLCI_DPO_TYPES)
        if pair is None:
            continue
        lc = n_chat_tokens(tokenizer, pair["prompt"] + pair["chosen"])
        lr = n_chat_tokens(tokenizer, pair["prompt"] + pair["rejected"])
        if max(lc, lr) > Q["dpo_max_len"]:
            continue
        pair.update(chosen_tokens = lc, rejected_tokens = lr)
        out.append(pair)
    print(f"Dolci DPO: kept {len(out)} of {scanned} scanned")
    return out

dolci_pairs = []
if DPO_DATA in ("dolci", "both"):
    dolci_pairs = load_dolci_dpo(Q["dolci_n"])
    dolci_pairs, DATA_INFO["dolci_dropped_decontam"] = decontam_rows(
        dolci_pairs, lambda r: {"prompt": r["prompt"], "completion": r["chosen"], "text": r["rejected"][0]["content"]},
        "dpo_dolci")
    from collections import Counter
    DATA_INFO["dolci_pairs"] = len(dolci_pairs)
    DATA_INFO["dolci_sources"] = Counter(r["source"].rsplit("-request", 1)[0] for r in dolci_pairs).most_common(10)
    DATA_INFO["dolci_models"] = Counter(f'{r["chosen_model"]} > {r["rejected_model"]}' for r in dolci_pairs).most_common(5)
    if dolci_pairs:   # 长度差：防止模型靠"写长"投机（OLMo 3 用长度归一化的 DPO 也是为此）
        DATA_INFO["dolci_mean_tokens"] = dict(chosen = round(sum(r["chosen_tokens"] for r in dolci_pairs) / len(dolci_pairs)),
                                              rejected = round(sum(r["rejected_tokens"] for r in dolci_pairs) / len(dolci_pairs)))
print(DATA_INFO)

Generating train split:  65%|▋| 817000/1253230 [00:04<00:05, 84995.50 examples/
Generating train split:  66%|▋| 829000/1253230 [00:04<00:04, 90706.19 examples/
Generating train split:  67%|▋| 840000/1253230 [00:04<00:04, 93709.63 examples/
Generating train split:  68%|▋| 851000/1253230 [00:05<00:04, 95132.68 examples/
Generating train split:  69%|▋| 863000/1253230 [00:05<00:03, 97677.94 examples/
Generating train split:  70%|▋| 876000/1253230 [00:05<00:03, 97638.40 examples/
Generating train split:  71%|▋| 889000/1253230 [00:05<00:03, 100156.97 examples
Generating train split:  72%|▋| 902000/1253230 [00:05<00:03, 99998.66 examples/
Generating train split:  73%|▋| 915000/1253230 [00:05<00:03, 91621.01 examples/
Generating train split:  74%|▋| 925000/1253230 [00:05<00:03, 88191.73 examples/
Generating train split:  75%|▋| 935000/1253230 [00:06<00:03, 89629.44 examples/
Generating train split:  75%|▊| 946000/1253230 [00:06<00:03, 90925.51 examples/
Generating train split:  77%|▊| 959000/1

README.md:   0%|          | 0.00/4.81k [00:00<?, ?B/s]

allenai/Dolci-Instruct-DPO columns: ['chosen', 'chosen_model', 'preference_type', 'prompt_id', 'rejected', 'rejected_model']
Dolci DPO: kept 6000 of 34651 scanned
{
  "n_gram": 13,
  "train_rows_checked": 6000,
  "train_rows_flagged": 0,
  "overlap_rate": 0.0,
  "flagged_row_indices": [],
  "gpqa_questions_indexed": 546,
  "train_source": "data/dpo_dolci.jsonl"
}

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
[decontam] dpo_dolci: dropped 0 of 6000 rows (13-gram overlap with GPQA)
{'onpolicy_dropped_decontam': 0, 'onpolicy_prompts': 1500, 'dolci_dropped_decontam': 0, 'dolci_pairs': 6000, 'dolci_sources': [('

In [10]:
def make_pairs(rows, lora_request, rnd):
    # 每题采样 K 个回答 -> (答对且收尾, 其余) 配对。只打印聚合数字（MMLU-Pro / SciQ 是公开数据，但也不必打印）
    sp = SamplingParams(n = Q["k"], temperature = 1.0, top_p = 1.0, max_tokens = Q["gen_max"], seed = 3407 + rnd,
                        stop_token_ids = STOP_IDS, stop = STOP_STRS)
    t0 = time.time()
    outs = model.fast_generate([chat_prompt(tokenizer, r["prompt"]) for r in rows], sampling_params = sp,
                               lora_request = lora_request)
    rng = random.Random(rnd)
    pairs, n_good, n_trunc, n_all, all_good, all_bad = [], 0, 0, 0, 0, 0
    for r, o in zip(rows, outs):
        good = [c for c in o.outputs if is_correct_and_stopped(c.text, r["answer"], c.finish_reason)]
        bad = [c for c in o.outputs if c not in good]
        n_good += len(good); n_all += len(o.outputs)
        n_trunc += sum(c.finish_reason == "length" for c in o.outputs)
        if not good:
            all_bad += 1; continue
        if not bad:
            all_good += 1; continue
        for _ in range(min(PAIRS_PER_PROMPT, len(good), len(bad))):
            ch, rj = good.pop(rng.randrange(len(good))), bad.pop(rng.randrange(len(bad)))
            pairs.append({"prompt": [{"role": "user", "content": r["prompt"]}],
                          "chosen": [{"role": "assistant", "content": ch.text}],
                          "rejected": [{"role": "assistant", "content": rj.text}],
                          "chosen_tokens": len(ch.token_ids), "rejected_tokens": len(rj.token_ids),
                          "rejected_truncated": rj.finish_reason == "length"})
    n = max(1, len(rows))
    info = dict(round = rnd, prompts = len(rows), pairs = len(pairs),
                sample_correct_and_stopped = round(100 * n_good / max(1, n_all), 1),
                sample_truncation_rate = round(100 * n_trunc / max(1, n_all), 1),
                frac_all_good = round(100 * all_good / n, 1), frac_all_bad = round(100 * all_bad / n, 1),
                mean_chosen_tokens = round(sum(p["chosen_tokens"] for p in pairs) / max(1, len(pairs))),
                mean_rejected_tokens = round(sum(p["rejected_tokens"] for p in pairs) / max(1, len(pairs))),
                rejected_truncated = round(100 * sum(p["rejected_truncated"] for p in pairs) / max(1, len(pairs)), 1),
                minutes = round((time.time() - t0) / 60, 1))
    print(info)
    if wandb.run is not None:
        for k, v in info.items():
            wandb.run.summary[f"pairs/r{rnd}/{k}"] = v
    return pairs, info

## 3. DPO 训练

`ref_model=None`：LoRA 训练时 TRL 用"关掉 adapter 的模型"当参考模型，也就是 `INIT_MODEL`，不用再占一份显存。
`config_kwargs` 只保留当前 TRL 版本的 `DPOConfig` 真有的参数（例如新版本删掉了某个字段，会打印"ignored"而不是报错）。

In [11]:
import numpy as np
from datasets import Dataset
from trl import DPOTrainer, DPOConfig

def _avg(log, key, part):
    vals = [h[key] for h in log if key in h]
    k = max(1, len(vals) // 5)
    return round(float(np.mean(vals[:k] if part == "first" else vals[-k:])), 4) if vals else None

# 每次运行这一格都从全新的 LoRA（= INIT_MODEL 本身）开始；已经完成的轮次（RESUME=True）从持久目录读回，不重训。
ROUND_INFO, DPO_STATS, lora_req = [], [], None
print("fresh LoRA:", restore_lora(model))
for rnd in range(ROUNDS):
    cur_dir = ADIR(f"dpo_lora_r{rnd}")
    if CACHE.has(f"dpo_r{rnd}") and os.path.exists(f"{cur_dir}/adapter_model.safetensors"):
        _saved = CACHE.load(f"dpo_r{rnd}")
        print(f"[i] 第 {rnd} 轮已经完成，读回 adapter：", restore_lora(model, cur_dir))
        DPO_STATS.append(_saved["stats"]); ROUND_INFO += _saved["pairs"]
        lora_req = model.load_lora(cur_dir)
        quick_eval(f"{rnd + 1}_dpo_r{rnd}", lora_req)
        continue
    train_rows, _round_pairs = [], []
    if DPO_DATA in ("onpolicy", "both"):
        pairs, info = make_pairs(mcq, lora_req, rnd)
        ROUND_INFO.append(info); _round_pairs.append(info)
        train_rows += pairs
    if DPO_DATA in ("dolci", "both") and rnd == 0:
        train_rows += dolci_pairs
    if not train_rows:
        print("[!] 没有可用的偏好对（全对或全错？）：换题库或检查 INIT_MODEL"); break
    random.Random(rnd).shuffle(train_rows)
    ds = Dataset.from_list([{k: r[k] for k in ("prompt", "chosen", "rejected")} for r in train_rows])

    loss_kw = dpo_loss_kwargs(DPOConfig, LOSS_TYPE, SFT_WEIGHT, DPO_BETA)   # 兼容不同 TRL 版本的写法
    print("DPO loss:", loss_kw)
    args = DPOConfig(**config_kwargs(
        DPOConfig,
        learning_rate = DPO_LR,
        per_device_train_batch_size = Q["dpo_batch"], gradient_accumulation_steps = Q["dpo_accum"],
        num_train_epochs = 1, warmup_ratio = 0.1, lr_scheduler_type = "linear", optim = "adamw_8bit",
        weight_decay = 0.0, logging_steps = 5, max_length = Q["dpo_max_len"], max_prompt_length = 1024,
        seed = 3407, report_to = REPORT_TO, run_name = f"{RUN_NAME}-r{rnd}", output_dir = f"outputs/dpo_r{rnd}",
        save_strategy = "no", **loss_kw))
    trainer = DPOTrainer(model = model, ref_model = None, args = args, train_dataset = ds, processing_class = tokenizer)
    stats = trainer.train()
    log = trainer.state.log_history
    s = dict(round = rnd, pairs = len(ds), steps = trainer.state.global_step, loss = loss_kw,
             minutes = round(stats.metrics["train_runtime"] / 60, 1))
    for key in ["loss", "rewards/margins", "rewards/accuracies", "rewards/chosen", "rewards/rejected",
                "logps/chosen", "logps/rejected", "nll_loss"]:
        if any(key in h for h in log):
            s[key] = {"first": _avg(log, key, "first"), "last": _avg(log, key, "last")}
    DPO_STATS.append(s)
    print(s)
    model.save_lora(cur_dir)                               # 直接存在持久目录
    CACHE.save(f"dpo_r{rnd}", {"stats": s, "pairs": _round_pairs})   # 这一轮完成：断线后不会重训
    lora_req = model.load_lora(cur_dir)
    quick_eval(f"{rnd + 1}_dpo_r{rnd}", lora_req)
    del trainer, ds
    torch.cuda.empty_cache()
REPORT.update(data = DATA_INFO, pairs = ROUND_INFO, dpo = DPO_STATS)

fresh LoRA: {'reset_layers': 252}
(MainProcess pid=9982) INFO 11-17 04:30:03 [scheduler.py:2691] setting pause state to UNPAUSED


Rendering prompts:   0%|          | 0/1500 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 6000/6000 [1:20:09<00:00,  1.25it/s, est. speed input: 206.35 toks/s, output: 861.33 toks/s]


{'round': 0, 'prompts': 1500, 'pairs': 1026, 'sample_correct_and_stopped': 68.9, 'sample_truncation_rate': 0.8, 'frac_all_good': 39.5, 'frac_all_bad': 8.0, 'mean_chosen_tokens': 709, 'mean_rejected_tokens': 830, 'rejected_truncated': 3.1, 'minutes': 80.2}
DPO loss: {'loss_type': 'sigmoid_norm', 'beta': 5.0}
[i] DPOConfig has no ['max_prompt_length', 'warmup_ratio']; ignored


Tokenizing train dataset (num_proc=6):   0%|          | 0/7026 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset (num_proc=6):   0%|          | 0/7026 [00:00<?, ? example…

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,026 | Num Epochs = 1 | Total steps = 440
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 132,120,576 of 4,286,709,248 (3.08% trained)


Unsloth: Will smartly offload gradients to save VRAM!
(MainProcess pid=9982) WARNING 11-17 05:51:23 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _rms_layernorm_forward. This causes a latency spike; consider extending warmup to cover this shape/config.
(MainProcess pid=9982) WARNING 11-17 05:51:24 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _fl_partial_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(MainProcess pid=9982) WARNING 11-17 05:51:24 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _fl_flash_p_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(MainProcess pid=9982) WARNING 11-17 05:51:24 [jit_monitor.py:140] Triton kernel JIT compilation during inference: _fl_combine_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(MainProcess pid=9982) WARNING 11-17 05:51:24 [jit_monitor.py:140] Tri

Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logits / chosen,logits / rejected,logps / chosen,logps / rejected
5,0.693079,0.029303,-0.233467,0.250000,0.262770,6.742852,6.878618,-578.810010,-437.218342
10,0.689898,0.167363,-3.905365,0.700000,4.072728,6.614747,6.537421,-710.692103,-473.924836


OutOfMemoryError: CUDA out of memory. Tried to allocate 270.00 MiB. GPU 0 has a total capacity of 22.03 GiB of which 109.12 MiB is free. Including non-PyTorch memory, this process has 21.90 GiB memory in use. Of the allocated memory 21.16 GiB is allocated by PyTorch, with 788.36 MiB allocated in private pools (e.g., CUDA Graphs), and 906.13 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 4. 汇总（W&B 上也有同样的表；最后打印的 JSON 可以贴回给 Claude）

In [ ]:
print("训练前后的配对比较（同一批题，后 − 前，单位：分）：")
COMPARE = EVAL.compare_all()

rows = print_table(RESULTS)
REPORT.update(quick_eval = RESULTS, compare = COMPARE, versions = VERSIONS, gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB,
              wandb = wandb.run.url if REPORT_TO == "wandb" else None)
os.makedirs("outputs", exist_ok = True)
json.dump(REPORT, open(f"outputs/report_{EXP_NAME}.json", "w"), indent = 1, default = str)
if REPORT_TO == "wandb":
    wandb.log({"gpqa_quick_eval": wandb.Table(columns = TABLE_COLS, data = rows)})
    art = wandb.Artifact(f"report-{RUN_NAME}", type = "report")
    art.add_file(f"outputs/report_{EXP_NAME}.json")
    wandb.log_artifact(art)
    wandb.finish()
if "PERSIST" in globals():   # 断开之后也能在 Drive 里找到
    json.dump(REPORT, open(f"{PERSIST}/report_{EXP_NAME}.json", "w"), indent = 1, default = str)
print("=" * 30, "把下面整段 JSON 贴回给 Claude", "=" * 30)
print(json.dumps(REPORT, ensure_ascii = False, default = str))

## 5. 合并保存 → 推到 Hub（下一步 GRPO 从这里开始）

`merged_16bit` 把 LoRA 合并进权重；保存的 tokenizer 带着模板和 eos=`<|endoftext|>`。设了 `PUSH_REPO` 就推到 HF 私有仓库，
然后在 notebook 11 里 `RUN_SFT=False`、`INIT_MODEL=PUSH_REPO` 跑 GRPO。正式评测用 `main` 的 `01`（生成预算 8192）；A100 上训练的模型提交时注明 A100（不参与排名）。

In [12]:
if CACHE.has("pushed"):
    print("[i] 已经推送 / 保存过：", CACHE.load("pushed"))
else:
    _where = PUSH_REPO
    try:
        if not PUSH_REPO:
            raise ValueError("PUSH_REPO is None")
        try:
            model.push_to_hub_merged(PUSH_REPO, tokenizer, save_method = "merged_16bit", private = True,
                                     token = os.environ.get("HF_TOKEN"))
        except TypeError:
            model.push_to_hub_merged(PUSH_REPO, tokenizer, save_method = "merged_16bit", token = os.environ.get("HF_TOKEN"))
        print(f"pushed: https://huggingface.co/{PUSH_REPO} → notebook 11: INIT_MODEL = \"{PUSH_REPO}\"")
    except Exception as e:
        _where = ADIR("merged_16bit")
        print("[!] 没有推到 HF：", type(e).__name__, str(e)[:200], "→ 合并模型存到", _where, "（notebook 11 的 INIT_MODEL 填这个路径）")
        model.save_pretrained_merged(_where, tokenizer, save_method = "merged_16bit")
    CACHE.save("pushed", {"model": _where})

Unsloth: Restored added_tokens_decoder metadata in darcyyao/qwen3-4b-d1-dpo/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...




Unsloth: Copying 2 files from cache to `darcyyao/qwen3-4b-d1-dpo`: 100%|██████████| 2/2 [00:00<00:00, 1650.33it/s]

... (10 outputs left out)

  ...0002-of-00002.safetensors:   0%|          |  603kB / 3.08GB            

Validating                    :           |   0%            



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:30<00:00, 75.15s/it]


Unsloth: Merge process complete. Saved to `/content/GPQA-eval/darcyyao/qwen3-4b-d1-dpo`
pushed: https://huggingface.co/darcyyao/qwen3-4b-d1-dpo → notebook 11: INIT_MODEL = "darcyyao/qwen3-4b-d1-dpo"


## 6. 断开 runtime

`AUTO_DISCONNECT=True` 时这一格在 60 秒后释放 runtime，不再消耗 CU。可以 Run all 之后离开：中途断线就重新打开、再 Run all，已经完成的评测和 DPO 轮次会直接跳过。

In [ ]:
end_runtime(AUTO_DISCONNECT)